# Lab 3 — Recurrent Neural Network (RNN)

A **recurrent neural network** processes a sequence one step at a time, keeping a
**hidden state** that carries information forward — a form of memory. Here we
feed each Fashion-MNIST image to the RNN **row by row** (28 steps of 28 numbers)
and classify it after the last row.

You'll build a simple RNN, train and evaluate it, and **visualize how the hidden
state evolves** as the network reads down the image.

> **Heads up:** plain RNNs struggle to remember information across long
> sequences (the "vanishing gradient" problem). The next lab uses an **LSTM**,
> which is designed to fix exactly this.

## 1. Setup and data

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision
from torchvision import datasets, transforms
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             confusion_matrix, ConfusionMatrixDisplay)

torch.manual_seed(0)
np.random.seed(0)

# Reuse the Fashion-MNIST already stored in the project's data/ folder if we can
# find it (the notebooks live in sub-folders), otherwise download into ./data.
_candidates = ["./data", "../data", "../../data"]
DATA_ROOT = next((p for p in _candidates
                  if os.path.isdir(os.path.join(p, "FashionMNIST"))), "./data")
print("Data folder:", DATA_ROOT)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
# Fashion-MNIST has 10 clothing classes.
CLASS_NAMES = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
               "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]
NUM_CLASSES = len(CLASS_NAMES)

### Load Fashion-MNIST (the easy way)

In [ ]:
# The "simple" way: torchvision gives us a ready-made Dataset object.
# transforms.ToTensor() converts a PIL image (0-255) to a float tensor in [0, 1]
# and adds the channel dimension -> shape [1, 28, 28].
transform = transforms.ToTensor()

train_full = datasets.FashionMNIST(root=DATA_ROOT, train=True,
                                   download=True, transform=transform)
test_set   = datasets.FashionMNIST(root=DATA_ROOT, train=False,
                                   download=True, transform=transform)

print("Training examples:", len(train_full))
print("Test examples:    ", len(test_set))

img, label = train_full[0]
print("One image tensor shape:", img.shape, "| label:", label, "=", CLASS_NAMES[label])

In [ ]:
# Let's look at a few images so we know what we're classifying.
fig, axes = plt.subplots(2, 5, figsize=(9, 4))
for ax, (img, label) in zip(axes.ravel(), train_full):
    ax.imshow(img.squeeze(), cmap="gray")
    ax.set_title(CLASS_NAMES[label], fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

### Load the data yourself: a custom `Dataset`

The same custom `Dataset` from the earlier labs. The reshape into a sequence
happens inside the model, so the dataset is unchanged.

In [ ]:
# TODO: complete the custom Dataset class.
# A Dataset needs three methods: __init__, __len__, __getitem__.

class FashionCustomDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        # TODO: return the number of samples
        pass

    def __getitem__(self, idx):
        image = self.images[idx]        # [28, 28] uint8
        label = int(self.labels[idx])
        # TODO: turn `image` into a float tensor in [0, 1] with shape [1, 28, 28].
        #   hint: torch.from_numpy(...).float().div(255.0).unsqueeze(0)
        # TODO: apply self.transform if it is not None
        # TODO: return (image, label)
        pass

train_images = train_full.data.numpy()
train_labels = train_full.targets.numpy()
test_images  = test_set.data.numpy()
test_labels  = test_set.targets.numpy()

custom_train = FashionCustomDataset(train_images, train_labels)
custom_test  = FashionCustomDataset(test_images,  test_labels)

img_c, label_c = custom_train[0]
print("Custom sample shape:", img_c.shape, "| label:", CLASS_NAMES[label_c])

### DataLoaders

In [ ]:
# We split the training data into train/validation and wrap everything in
# DataLoaders. A DataLoader batches the data and shuffles it each epoch.
BATCH_SIZE = 128

val_size   = 10_000
train_size = len(custom_train) - val_size
train_ds, val_ds = random_split(custom_train, [train_size, val_size],
                                generator=torch.Generator().manual_seed(0))

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(custom_test, batch_size=BATCH_SIZE, shuffle=False)

print(f"train batches: {len(train_loader)}, "
      f"val batches: {len(val_loader)}, test batches: {len(test_loader)}")

# Every batch looks the same no matter which model we build:
xb, yb = next(iter(train_loader))
print("batch images:", xb.shape, "| batch labels:", yb.shape)

### Treating an image as a sequence

RNNs and LSTMs are built for **sequences** (words, time series, ...). A
Fashion-MNIST image is 28x28, so we can read it **one row at a time**: a sequence
of 28 time-steps, where each step is a 28-number row. The network reads top to
bottom and makes a prediction after the final row.

We reshape `[B, 1, 28, 28] -> [B, 28, 28]` **inside the model's forward()**, so
the data-loading and training code stays exactly the same as the earlier labs.

## 2. Build the RNN

In [ ]:
# TODO: build a RNN classifier that reads each image as 28 rows of 28 numbers.
# Suggested pieces:
#   self.rnn = nn.RNN(input_size=28, hidden_size=128, num_layers=1, batch_first=True)
#   self.fc  = nn.Linear(128, num_classes)
# In forward():
#   reshape x from [B,1,28,28] to [B, 28, 28]  (hint: x.view(x.size(0), 28, 28))
#   run it through self.rnn -> out, _
#   take the last time-step: out[:, -1, :]
#   pass that through self.fc and return the logits
class RNNClassifier(nn.Module):
    def __init__(self, input_size=28, hidden_size=128, num_layers=1, num_classes=10):
        super().__init__()
        # TODO: define self.rnn and self.fc
        pass

    def forward(self, x):
        # TODO: reshape, run the recurrent layer, take the last step, classify
        pass

model = RNNClassifier(num_classes=NUM_CLASSES)
print(model)

## 3. Train and validate

In [ ]:
# These three helpers work for EVERY model in this lab, because each model's
# forward() method reshapes the [batch, 1, 28, 28] input as it needs.

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()           # clear old gradients
        outputs = model(images)         # forward pass -> logits [batch, 10]
        loss = criterion(outputs, labels)
        loss.backward()                 # backprop
        optimizer.step()                # update weights

        running_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total   += labels.size(0)
    return running_loss / total, correct / total


@torch.no_grad()                        # no gradients needed for evaluation
def evaluate(model, loader, criterion):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(1)
        correct += (preds == labels).sum().item()
        total   += labels.size(0)
        all_preds.append(preds.cpu())
        all_labels.append(labels.cpu())

    avg_loss = running_loss / total
    accuracy = correct / total
    preds  = torch.cat(all_preds).numpy()
    labels = torch.cat(all_labels).numpy()
    return avg_loss, accuracy, preds, labels

In [ ]:
def fit(model, epochs=5, lr=1e-3):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion)
        va_loss, va_acc, _, _ = evaluate(model, val_loader, criterion)
        history["train_loss"].append(tr_loss)
        history["train_acc"].append(tr_acc)
        history["val_loss"].append(va_loss)
        history["val_acc"].append(va_acc)
        print(f"Epoch {epoch:2d}/{epochs} | "
              f"train loss {tr_loss:.3f} acc {tr_acc:.3f} | "
              f"val loss {va_loss:.3f} acc {va_acc:.3f}")
    return history

In [ ]:
model = RNNClassifier(num_classes=NUM_CLASSES)
history = fit(model, epochs=5, lr=1e-3)

## 4. Loss and accuracy curves

In [ ]:
def plot_curves(history):
    epochs = range(1, len(history["train_loss"]) + 1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

    ax1.plot(epochs, history["train_loss"], "o-", label="train")
    ax1.plot(epochs, history["val_loss"],   "o-", label="val")
    ax1.set_title("Loss"); ax1.set_xlabel("epoch"); ax1.legend()

    ax2.plot(epochs, history["train_acc"], "o-", label="train")
    ax2.plot(epochs, history["val_acc"],   "o-", label="val")
    ax2.set_title("Accuracy"); ax2.set_xlabel("epoch"); ax2.legend()

    plt.tight_layout(); plt.show()

plot_curves(history)

## 5. Evaluate: accuracy, precision, recall

In [ ]:
# Run the model on the held-out test set, then compute standard metrics.
criterion = nn.CrossEntropyLoss()
test_loss, test_acc, preds, labels = evaluate(model, test_loader, criterion)

# "macro" averaging treats every class equally (good for balanced data).
precision = precision_score(labels, preds, average="macro", zero_division=0)
recall    = recall_score(labels, preds, average="macro", zero_division=0)

print(f"Test loss     : {test_loss:.3f}")
print(f"Test accuracy : {test_acc:.3f}")
print(f"Precision(macro): {precision:.3f}")
print(f"Recall(macro)   : {recall:.3f}")

### Confusion matrix

In [ ]:
# A confusion matrix shows which classes get mixed up with which.
cm = confusion_matrix(labels, preds)
fig, ax = plt.subplots(figsize=(8, 7))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)
disp.plot(ax=ax, cmap="Blues", xticks_rotation=45, colorbar=False)
ax.set_title("Confusion matrix (test set)")
plt.tight_layout(); plt.show()

## Visualizing the hidden state over time

The RNN produces a hidden-state vector after **every** row it reads. Stacking
those vectors gives a picture of how the network's "memory" changes as it scans
the image from top to bottom. Each column below is one time-step (one row of the
image); each row is one hidden unit.

In [ ]:
# Grab the full sequence of hidden states for one image.
model.eval()
sample_img, sample_label = custom_test[0]
seq = sample_img.view(1, 28, 28).to(device)     # [1, 28 steps, 28 feats]
with torch.no_grad():
    all_hidden, _ = model.rnn(seq)              # [1, 28, hidden_size]
all_hidden = all_hidden[0].cpu().numpy().T      # -> [hidden_size, 28 steps]

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(11, 4),
                               gridspec_kw={"width_ratios": [1, 3]})
ax0.imshow(sample_img.squeeze(), cmap="gray")
ax0.set_title(f"input: {CLASS_NAMES[sample_label]}"); ax0.axis("off")

im = ax1.imshow(all_hidden, aspect="auto", cmap="coolwarm")
ax1.set_title("RNN hidden state as it reads the image")
ax1.set_xlabel("time-step (image row, top -> bottom)")
ax1.set_ylabel("hidden unit")
fig.colorbar(im, ax=ax1, fraction=0.025)
plt.tight_layout(); plt.show()

print("Notice how the hidden state settles as more of the image is read.")

## 6. Your turn

1. Increase `hidden_size` or set `num_layers=2`. Does accuracy improve?
2. Read the image **column by column** instead of row by row. Any difference?
3. Compare the RNN with the LSTM in the next lab — which trains more smoothly?